# Mediating Faith and Power: DNBLab Online Streaming LDA Workbench (1970–2025)
**Doctoral Dissertation Research Observatory — Centre for German Studies, Jawaharlal Nehru University (JNU)**  
**Scholar:** Paul Marandi  
**Corpus Scale:** `10,545,315` Analytical Passages · `10,042,681` Unique German Works · `19,914` Complete Full-Text Books · `44,339` DNBLab § 60d UrhG Commercial Print & E-Book Passages (`dnblab_dtf_incopyright`, `29.90M` Words) · `166,290` Digitized TOCs (`/04/text` & PDFs) · `192,814+` Publisher Blurbs · `831,286,411` German Words across `126` Zstd-19 Apache Parquet Shards (`1,575.59 MB`)

---

### Dual Execution Modes Supported by This DNBLab Notebook
1. **Mode A — Public OpenData + DNBLab Commercial Print & E-Book Incorporator (`--dnblab-incopyright-commercial`):**
   - Streams all `126` Zstd-19 Apache Parquet shards (`data/parquet/dnb_corpus_part001.parquet` .. `dnb_corpus_part126.parquet`, automatically hydrating any missing shards from GitHub Release [`v12.0-dnb-parquet-shards`](https://github.com/metachasm-beep/Deutsche-Verlage-LDA/releases/tag/v12.0-dnb-parquet-shards)).
   - Incorporates In-Copyright Commercial Print & E-Books (`Suhrkamp`, `Herder`, `C.H. Beck`, `Fischer`, `Rowohlt`, `Kösel`, `Gräfe und Unzer`, `Vandenhoeck & Ruprecht`, `Mohr Siebeck`, `Kohlhammer`, `Campus`, `Transcript`, `De Gruyter`, `Patmos`, `Kreuz`, `Goldmann`, `Droemer Knaur`, `Schöningh`, `Böhlau`, `Evangelische Verlagsanstalt`, `Gütersloher Verlagshaus`, `Aschendorff`, `Klett-Cotta`, `Springer`, `Pustet`, `St. Benno`, `Echter`, `Schwabenverlag`, `Alibri`, `Wagenbach`, `Kiepenheuer & Witsch`, `Meiner`, `Metzler`) via `dnb_harvester.incorporate_dnblab_incopyright_commercial_books()` (`REIHE_O_LF` `/34` e-books + concurrent dual `/blurb/` + `/04/text` fusion into `dnblab_dtf_incopyright`).
2. **Mode B — On-Site DNB Leipzig / Frankfurt DNBLab Workstation (§ 60d UrhG Text & Data Mining):**
   - For copyright-protected (`access_restricted`) digital deposits mounted inside DNB's on-site virtual machines under **§ 60d UrhG**, Cell 4 streams restricted PDF/EPUB files in RAM, converts them into non-reconstructible **Derived Text Formats (DTFs)** (`doc2bow` token-frequency bags + `|V| = 25,000` lemma/bigram dictionary), updates the global Online Variational Bayes posterior via `lda_model.update(dtf_bow_batch)`, and exports copyright-compliant model JSONs (`app/lda_topics_report.json`, `app/lda_modes_report.json`) without exporting copyrighted raw text.

In [ ]:
import os
import sys
import json
import glob
from pathlib import Path

# Ensure repository root is on sys.path whether launched from root or notebooks/
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import config
import lda_pipeline
from scripts import dnb_harvester

fair_env = lda_pipeline.collect_fair_ddi_environment()
print('=== DNBLab FAIR / DDI Computational Environment ===')
print(json.dumps(fair_env, indent=2, ensure_ascii=False))

## 1. Verify or Hydrate the 126 Zstd-19 Parquet Shards (`10,519,923` Passages)
If running on a fresh DNBLab virtual machine or cloud instance without local shards in `data/parquet/`, `ensure_parquet_shards_from_github_release()` automatically streams `dnb_corpus_part001.parquet` .. `dnb_corpus_part126.parquet` and `parquet_manifest.json` from GitHub Release `v12.0-dnb-parquet-shards` and verifies their SHA-256 digests.

In [ ]:
PARQUET_DIR = 'data/parquet'
shard_paths = lda_pipeline.ensure_parquet_shards_from_github_release(
    parquet_dir=PARQUET_DIR,
    release_tag='v12.0-dnb-parquet-shards',
    repo='metachasm-beep/Deutsche-Verlage-LDA',
)

manifest_path = os.path.join(PARQUET_DIR, 'parquet_manifest.json')
with open(manifest_path, 'r', encoding='utf-8') as mf:
    manifest = json.load(mf)

print(f"Verified {len(shard_paths)} Zstd-19 Parquet shards in `{PARQUET_DIR}`:")
print(f"  • Total Analytical Passages : {manifest['total_rows']:,}")
print(f"  • Total German Words        : {manifest['total_words']:,}")
print(f"  • Compressed Size (Zstd-19) : {manifest['total_parquet_mb']:,.2f} MB")

## 2. DNBLab Live Endpoint Hydration (`/04/text` TOC OCRs, `/blurb/`, & `onlinefree` `/34` OA Books)
Uses our hybrid `aiohttp` dual-semaphore engine (`_async_hydrate_shard_blurbs_and_tocs`, ~110+ req/s) + `ThreadPoolExecutor` PDF/EPUB RAM streamer (`stream_dnb_oa_multi_url_in_ram`) to hydrate additional DNB `MARC 856 $u` links directly into the Zstd-19 Parquet shards.

In [ ]:
# Load or refresh the 1,295,009 guaranteed-free (`onlinefree`) DNB /34 IDNs from DNBLab OpenData dumps
free_idns = dnb_harvester.load_or_fetch_dnblab_free_idns()
print(f'Indexed {len(free_idns):,} guaranteed-free (`onlinefree`) DNBLab /34 full-text IDNs.')

# Optional: Run an incremental hydration pass on selected Parquet shards
# hydration_stats = dnb_harvester.hydrate_parquet_shards_with_remote_fulltext_and_blurbs(
#     parquet_dir=PARQUET_DIR,
#     start_shard=8,
#     end_shard=125,
#     max_oa_per_shard=15,
#     max_toc_per_shard=600,
#     max_blurbs_per_shard=650,
# )

## 3. § 60d UrhG On-Site DNBLab Derived Text Format (DTF) Streamer (For Restricted In-Copyright `/34` Deposits)
When executing inside DNB's on-site workstation in Leipzig or Frankfurt am Main with access to local restricted digital objects (`DNBLAB_LOCAL_OBJECT_ROOT`), this helper extracts lemmas and compounds in volatile RAM and emits **only non-reconstructible Bag-of-Words (`doc2bow`) token-frequency vectors** (Derived Text Format / DTF compliant with § 60d UrhG).

In [ ]:
def stream_dnblab_restricted_folder_to_dtf_bow(
    local_object_dir: str,
    dictionary,
    bigram_phraser=None,
    chunk_words: int = 1000,
) -> list[dict]:
    """
    § 60d UrhG Derived Text Format (DTF) converter for on-site DNBLab workstations:
    Reads local PDF/EPUB/TXT files from `local_object_dir`, lemmatizes & decompounds in RAM,
    discards raw text immediately, and returns irreversible (token_id, count) BoW vectors.
    """
    import fitz
    dtf_records = []
    if not os.path.exists(local_object_dir):
        print(f'[Info] On-site restricted directory `{local_object_dir}` not mounted; using public + hydrated Parquet shards.')
        return dtf_records

    for root, _, files in os.walk(local_object_dir):
        for fname in files:
            if not fname.lower().endswith(('.pdf', '.epub', '.txt')):
                continue
            fpath = os.path.join(root, fname)
            try:
                if fname.lower().endswith('.txt'):
                    raw_text = Path(fpath).read_text(encoding='utf-8', errors='ignore')
                else:
                    with fitz.open(fpath) as doc:
                        raw_text = ' '.join(page.get_text('text') for page in doc)
                words = raw_text.split()
                del raw_text  # Discard copyrighted prose from RAM immediately
                for ch_i, idx in enumerate(range(0, len(words), chunk_words), start=1):
                    chunk_str = ' '.join(words[idx : idx + chunk_words])
                    if len(chunk_str.split()) < 80:
                        continue
                    lemmas = lda_pipeline.preprocess_text(chunk_str)
                    cleaned = lda_pipeline.clean_row_tokens_for_modeling(lemmas, {'title': fname})
                    if bigram_phraser is not None:
                        cleaned = [t for t in bigram_phraser[cleaned] if t not in lda_pipeline.ALL_STOPWORDS]
                    bow = dictionary.doc2bow(cleaned)
                    if bow:
                        dtf_records.append({'object_id': fname, 'chunk_index': ch_i, 'bow': bow})
            except Exception:
                continue
    return dtf_records

print('§ 60d UrhG Derived Text Format (DTF) converter ready.')

## 4. Run Online Variational Bayes Streaming LDA Over All 126 Parquet Shards (`10,519,923` Passages)
Executes the 2-pass Hoffman, Blei & Bach (2010) Online Streaming LDA pipeline (`lda_pipeline.run_streaming_online_lda_pipeline`) in `< 1.2 GB` RAM across `data/parquet/`, training the $K = 12$ Meso-Inflection topic model (`|V| = 25,000` lemmas/bigrams), computing 5-fold cross-validated sLDA diagnostics, and exporting directly to:
- `app/lda_topics_report.json`
- `app/lda_modes_report.json`
- `app/lda_viz.html`
- `app/data_summary.json`

In [ ]:
lda_model, sample_corpus, dictionary, coherence_score, representative_docs = (
    lda_pipeline.run_streaming_online_lda_pipeline(
        data_path=PARQUET_DIR,
        num_topics=12,
        lda_mode='unsupervised',
        export_to_app=True,
    )
)

print(f'\nTrained Online Streaming LDA Model:')
print(f'  • Total Streamed Passages : {getattr(lda_model, "total_streamed_docs", 0):,}')
print(f'  • Vocabulary Size (|V|)   : {len(dictionary):,}')
print(f'  • Coherence (U_Mass)      : {coherence_score:.4f}')

## 5. Inspect Posterior Topics ($K = 12$), Ideological Regimes ($\text{RQ}_1$–$\text{RQ}_3$), & Diachronic Trajectory (1970s–2020s)

In [ ]:
with open('app/lda_topics_report.json', 'r', encoding='utf-8') as rf:
    report = json.load(rf)

for t in report['topics']:
    kws = ', '.join(f"{k['term']} ({k['weight']:.3f})" for k in t['top_terms'][:8])
    pubs = ', '.join(f"{p['publisher']} ({p['share']}%)" for p in t['top_publishers'][:3])
    print(f"Topic {t['topic_id']:02d} [{t['archetype']['label']} · {t['archetype']['confidence']}%]")
    print(f"  Keywords  : {kws}")
    print(f"  Publishers: {pubs}")

print('\n=== Diachronic Decade Trajectory (1970s–2020s) ===')
for dec, shares in report.get('diachronic_decade_shares', {}).items():
    top3 = sorted(shares.items(), key=lambda x: x[1], reverse=True)[:3]
    print(f"  {dec}: {', '.join(f'{k}={v}%' for k, v in top3)}")